# Work Package 1 — Combined Single Tissue Channel × Exp-5 (Residual Fusion, tissue_only) — Colab Run

Advisor's own note: **"Tek kanal exp-4 ve exp-5 denenecek"** — 5 kanal yerine tum
dokular tek kanalda verilsin (all five tissue channels collapsed into ONE channel,
instead of concatenating them separately).

This notebook trains exp-5's `tissue_only` residual-fusion architecture with the
tissue arm seeing a **single combined channel** = per-pixel `max(granulation,
slough, necrosis, tendon, bone)`. `z_final = z_rgb + delta_z`: `z_rgb` is loaded from
task_3's own **frozen**, already-trained per-fold `mit_b1__corn` baseline checkpoint
— **never retrained here**. `delta_z` comes from a residual head that sees only the
combined tissue channel. Before training starts for every fold this notebook's
`train.py` re-verifies `z_final == z_rgb` and `delta_z == 0` at construction against
real data — the same two-step check exp-5's own `train.py` does.

`config_id = mit_b1__corn__wp1_exp5tissueonly__tissue=combined_max`, one row per fold
in `results.csv` (merged by fold key so a rerun overwrites rather than duplicates).

**Reference numbers to compare against** (already recorded, not re-derived here):
- RGB baseline (task_3, 3-fold mean): accuracy 0.8168, macro-F1 0.8182, QWK 0.9138
- Exp-5 `tissue_only`, all 5 channels separately fused, **fold 1 only** (3-fold
  sweep not completed in the base experiment): accuracy 0.8370, macro-F1 0.8376,
  QWK 0.9323 — notably close to that fold's own RGB baseline (0.8370/0.8369/0.9332),
  consistent with exp-5's diagnostic finding that the residual correction is close
  to content-independent even in the "fixed" `tissue_only` architecture.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Same `wound-staging-masks-exp-3.zip` as exp-3/exp-4/exp-5.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-3"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/task_3/requirements.txt

## 5. RGB baseline checkpoints (fold0/1/2) — required, not retrained here

This WP's `train.py` hard-fails (`SystemExit`) per fold if
`task_3/results/task_3_outputs/checkpoints/mit_b1__corn_fold{fold}.pt` is missing —
it is never retrained inside this experiment, only loaded and frozen. First tries
to copy the 3 files from `drive/MyDrive/thesis/task_3_outputs/` (task_3's own Colab
notebook output location); only whichever folds are still missing after that get
trained, using task_3's own unmodified `run_ablation.py --configs mit_b1__corn`
(fast: ~90s/fold).

In [ ]:
import glob
import os
import shutil

os.chdir(f"{REPO_DIR}/task_3")
CKPT_DIR = "results/task_3_outputs/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DRIVE_CKPT_DIR = "/content/drive/MyDrive/thesis/task_3_outputs"
all_drive_pt_files = glob.glob(f"{DRIVE_CKPT_DIR}/**/*.pt", recursive=True)
print(f"{len(all_drive_pt_files)} .pt file(s) found under {DRIVE_CKPT_DIR}")

for fold in range(3):
    local_path = f"{CKPT_DIR}/mit_b1__corn_fold{fold}.pt"
    if os.path.exists(local_path):
        continue
    candidates = [
        p for p in all_drive_pt_files
        if "mit_b1" in p.lower() and "corn" in p.lower() and "softmax" not in p.lower()
        and f"fold{fold}" in p.lower().replace("_fold", "fold").replace("-fold", "fold")
    ]
    if candidates:
        shutil.copy(candidates[0], local_path)
        print(f"fold{fold}: copied from Drive -> {candidates[0]}")
    else:
        print(f"fold{fold}: no match found under {DRIVE_CKPT_DIR}")

missing = [f for f in range(3) if not os.path.exists(f"{CKPT_DIR}/mit_b1__corn_fold{f}.pt")]
print("still missing after checking Drive:", missing)

if missing:
    !python check_compatibility.py
    !python run_ablation.py --configs mit_b1__corn --batch 16
else:
    print("all 3 RGB baseline checkpoints present, skipping training")

## 6. cd into this WP's scripts directory

In [ ]:
os.chdir(f"{REPO_DIR}/more_experiments/experiment_extended/wp1_single_tissue_channel/exp5_tissue_only/scripts")
print(os.getcwd())

## 6b. Smoke-test sanity check

Exercises the baseline-load + `verify_reproduces_baseline` check end-to-end on one
fold before committing to the full 3-fold run.

In [ ]:
!python train.py --folds 0 --smoke --workers 2

## 7. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/wp1_exp5tissueonly_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 8. Train — 3 folds, combined-max tissue channel

Per exp-5's own policy, this is **not** wrapped in a blind-retry loop — a crash
(checkpoint mismatch, OOM, failed baseline-reproduction check) means investigate,
not auto-restart. `train.py` always retrains whatever fold(s) you pass it (no
internal skip-if-done), so this cell checks `results.csv` itself first and only
asks `train.py` for the folds that are still missing via `--folds`.

In [ ]:
import os
import subprocess

import pandas as pd

results_path = f"{OUTPUT_DIR}/results/results.csv"
config_id = "mit_b1__corn__wp1_exp5tissueonly__tissue=combined_max"

if os.path.exists(results_path):
    existing = pd.read_csv(results_path)
    done_folds = set(existing.loc[existing["config_id"] == config_id, "fold"].astype(int))
else:
    done_folds = set()

missing_folds = sorted({0, 1, 2} - done_folds)
if not missing_folds:
    print("all 3 folds already in results.csv, nothing to train")
else:
    print(f"training folds: {missing_folds}")
    proc = subprocess.run(["python", "-u", "train.py", "--folds", ",".join(map(str, missing_folds)), "--workers", "4"])
    if proc.returncode != 0:
        print(f"FAILED with exit code {proc.returncode} -- investigate before re-running this cell")
    else:
        print("done")

## 9. Check results

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())

print("\nReference (already recorded, not re-derived here):")
print("  RGB baseline (3-fold mean):                 accuracy=0.8168 macro_f1=0.8182 qwk=0.9138")
print("  Exp-5 tissue_only, 5 channels separate, fold1: accuracy=0.8370 macro_f1=0.8376 qwk=0.9323")